# BSM Reduced-Form Model: Quickstart

This notebook loads the released model, predicts all 23,495 outputs for five
example scenarios, and reshapes the flat predictions into labelled time series.
The final section shows how to run an archived model version from `versions/`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

from bsm_public_rf import load_model, output_catalog, unpack_outputs

ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()

## 1. Load the model and inputs

`example_inputs.csv` holds five synthetic scenarios drawn uniformly within each
input's historical range. They are illustrative, not meaningful BSM scenarios.
To use your own inputs, provide a CSV with one column per required input; see
`model.input_schema()` for names, units, and ranges.

In [ ]:
model = load_model()
inputs = pd.read_csv(ROOT / "examples" / "example_inputs.csv", index_col="scenario")

print(
    f"{len(model.required_input_names)} inputs -> {len(model.feature_names)} features "
    f"-> {len(model.output_names)} outputs"
)
model.input_schema().head()

## 2. Predict

In [ ]:
predictions = model.predict(inputs)
predictions.iloc[:, :5]

## 3. Understand the outputs

Output names follow `VARIABLE[pathway, region, product]_YEAR`. `output_catalog`
splits each name and attaches its units and description from
`model/output_metadata.json`.

In [ ]:
catalog = output_catalog(model.output_names)
catalog.drop_duplicates("variable")[["unit", "description"]]

## 4. Unpack into time series

`unpack_outputs` gives one row per scenario and series (635 series per scenario)
with one column per year. Region codes are expanded to full names. Use bracket
indexing (`df["sample"]`), because `sample` and `product` are also DataFrame methods.

In [ ]:
unpacked = unpack_outputs(predictions)
print(unpacked.shape)

unpacked[
    (unpacked["sample"] == "scenario_1")
    & (unpacked["variable"] == "AHC.MFSPMetric")
    & (unpacked["pathway"] == "HEFA")
    & (unpacked["region"] == "Atlantic")
]

## 5. Archived model versions

`versions/` holds earlier model fits in their original format (MathJSON feature
definitions and Parquet coefficients). They have different inputs from the released
model but the same outputs, so the same unpacking applies. `BSM_RFM_v1` uses 62 inputs;
here each is set to the midpoint of its documented range.

In [ ]:
sys.path.insert(0, str(ROOT / "versions"))
import bsm_model_utils as v1  # noqa: E402

v1_inputs_metadata = json.loads(
    (ROOT / "versions" / "BSM_RFM_v1" / "inputs_metadata.json").read_text(encoding="utf-8")
)
v1_inputs = pd.DataFrame(
    [
        {
            name: (m["constraints"]["min"] + m["constraints"]["max"]) / 2
            for name, m in v1_inputs_metadata.items()
        }
    ]
)
v1_path = ROOT / "examples" / "v1_midpoint_inputs.csv"
v1_inputs.to_csv(v1_path, index=False)

v1_predictions = v1.run_pipeline(v1_path, data_dir=ROOT / "versions")
v1_path.unlink()
v1_predictions.iloc[:, :5]